In [0]:
WITH ae_accounts AS (
  SELECT DISTINCT deployable_account_name
  FROM main.gtm_gold.account_consumption_daily
  WHERE horizontal_and_vertical_hierarchy_concatenated_emails LIKE CONCAT('%', :ae_email, '%')
    AND YEAR(usage_date) + CASE WHEN MONTH(usage_date) >= 2 THEN 1 ELSE 0 END >= 2026
),
filtered_accounts AS (
  -- Same AE + Business_Unit + region filters the original query applied directly on account_consumption_daily
  -- for the DBU CTE. Kept sourced from account_consumption_daily since aibi_account_daily_consumption has
  -- no AE-email / Business_Unit / sales_subregion columns.
  SELECT DISTINCT deployable_account_name
  FROM main.gtm_gold.account_consumption_daily
  WHERE horizontal_and_vertical_hierarchy_concatenated_emails LIKE CONCAT('%', :ae_email, '%')
    AND Business_Unit = :business_unit
    AND sales_subregion_level_1 = :region_level_1
    AND sales_subregion_level_2 = :region_level_2
    AND YEAR(usage_date) + CASE WHEN MONTH(usage_date) >= 2 THEN 1 ELSE 0 END >= 2026
),
genie_kpis AS (
  SELECT
    g.date,
    h.deployable_account_name,
    SUM(g.genie_users_t7d_account_level)  AS genie_t7d_users,
    SUM(g.genie_users_t28d_account_level) AS genie_t28d_users
  FROM main.field_emea_product_usage.aibi_account_daily_consumption g
  INNER JOIN main.fin_live_gold.sfdc_hierarchy_mapping h
    ON g.sfdcAccountName = h.sfdc_account_name
  INNER JOIN ae_accounts a
    ON h.deployable_account_name = a.deployable_account_name
  WHERE YEAR(g.date) + CASE WHEN MONTH(g.date) >= 2 THEN 1 ELSE 0 END >= 2026
  GROUP BY g.date, h.deployable_account_name
),
genie_dbu_raw AS (
  SELECT
    g.date AS usage_date,
    h.deployable_account_name,
    SUM(g.genie_dollars_t1d)       AS genie_dbu_dollars,
    SUM(g.genie_dollars_t7d)       AS genie_t7d_dbu_dollars,
    SUM(g.genie_dollars_t28d)      AS genie_t28d_dbu_dollars,
    SUM(g.genie_dollars_t28d_lag28) AS genie_t28d_dbu_dollars_prev
  FROM main.field_emea_product_usage.aibi_account_daily_consumption g
  INNER JOIN main.fin_live_gold.sfdc_hierarchy_mapping h
    ON g.sfdcAccountName = h.sfdc_account_name
  INNER JOIN filtered_accounts fa
    ON h.deployable_account_name = fa.deployable_account_name
  WHERE YEAR(g.date) + CASE WHEN MONTH(g.date) >= 2 THEN 1 ELSE 0 END >= 2026
  GROUP BY g.date, h.deployable_account_name
),
genie_dbu AS (
  -- aibi_account_daily_consumption only carries a 28-day lag for $, not a 7-day lag, so the
  -- t7d "_prev" (value 7 days earlier, verified against account_consumption_daily's own
  -- _t7d_sum_prev convention) is computed here via LAG(...,7) over the daily series.
  SELECT
    usage_date,
    deployable_account_name,
    genie_dbu_dollars,
    genie_t7d_dbu_dollars,
    LAG(genie_t7d_dbu_dollars, 7) OVER (PARTITION BY deployable_account_name ORDER BY usage_date) AS genie_t7d_dbu_dollars_prev,
    genie_t28d_dbu_dollars,
    genie_t28d_dbu_dollars_prev
  FROM genie_dbu_raw
),
fy_start AS (
  -- First date available in aibi_account_daily_consumption >= start of the FY containing the latest snapshot
  SELECT MIN(date) AS fy_first_date
  FROM main.field_emea_product_usage.aibi_account_daily_consumption
  WHERE date >= MAKE_DATE(
    CASE WHEN MONTH((SELECT MAX(date) FROM main.field_emea_product_usage.aibi_account_daily_consumption)) >= 2
         THEN YEAR((SELECT MAX(date) FROM main.field_emea_product_usage.aibi_account_daily_consumption))
         ELSE YEAR((SELECT MAX(date) FROM main.field_emea_product_usage.aibi_account_daily_consumption)) - 1
    END, 2, 1)
),
fy_start_kpis AS (
  -- T28D users per account on the first day of the current FY
  SELECT
    h.deployable_account_name,
    SUM(g.genie_users_t28d_account_level) AS fy_start_t28d_users
  FROM main.field_emea_product_usage.aibi_account_daily_consumption g
  INNER JOIN main.fin_live_gold.sfdc_hierarchy_mapping h
    ON g.sfdcAccountName = h.sfdc_account_name
  INNER JOIN ae_accounts a
    ON h.deployable_account_name = a.deployable_account_name
  CROSS JOIN fy_start fs
  WHERE g.date = fs.fy_first_date
  GROUP BY h.deployable_account_name
)

SELECT
  k.date,
  YEAR(k.date) + CASE WHEN MONTH(k.date) >= 2 THEN 1 ELSE 0 END AS fiscal_year,
  CONCAT(
    'FY', RIGHT(CAST(YEAR(k.date) + CASE WHEN MONTH(k.date) >= 2 THEN 1 ELSE 0 END AS STRING), 2),
    '-',
    CASE
      WHEN MONTH(k.date) IN (2,3,4)   THEN 'Q1'
      WHEN MONTH(k.date) IN (5,6,7)   THEN 'Q2'
      WHEN MONTH(k.date) IN (8,9,10)  THEN 'Q3'
      ELSE                                 'Q4'
    END
  )  AS fiscal_quarter,
  CASE WHEN k.date = (SELECT MAX(date) FROM genie_kpis) THEN 'Y' ELSE 'N' END AS latest_snapshot,
  k.deployable_account_name,
  k.genie_t7d_users,
  k.genie_t28d_users,
  k.genie_t28d_users - f.fy_start_t28d_users as t28d_users_ytd_diff,
  fy_start_t28d_users,
  d.genie_dbu_dollars,
  d.genie_t7d_dbu_dollars,
  d.genie_t7d_dbu_dollars_prev,
  d.genie_t28d_dbu_dollars,
  d.genie_t28d_dbu_dollars_prev
FROM genie_kpis k
LEFT JOIN genie_dbu d
  ON  k.deployable_account_name = d.deployable_account_name
  AND k.date = d.usage_date
LEFT JOIN fy_start_kpis f
  ON  k.deployable_account_name = f.deployable_account_name
WHERE (DAYOFWEEK(k.date) = 6                          -- Fridays only
   OR k.date = (SELECT MAX(date) FROM genie_kpis))    -- plus latest available day
ORDER BY k.date, k.deployable_account_name